# **Demand uncertainty and hydraulic performance**

**Engineering question:** How does uncertainty in water demand affect pressure and system operation?

We will use the Net3 hydraulic model as an example.

First, we will use Python to automatically extract network information and simulation results. Then, we will modify demand at a single junction and compare the results with the baseline simulation. Finally, you will work in pairs to investigate the effects of demand uncertainty across the entire network and evaluate the engineering implications.

## 1. Load Net3 and run the baseline simulation

In [ ]:
try:
    import wntr
except ImportError:
    %pip install wntr
    import wntr
import numpy as np
import matplotlib.pyplot as plt

WNTR uses SI units. Adjust the file path if needed.

In [ ]:
inp = 'Net3.inp'  # change path if needed
wn = wntr.network.WaterNetworkModel(inp)
sim = wntr.sim.EpanetSimulator(wn)
results_base = sim.run_sim()
units = 15850.3  # m3/s to gpm

## 2. Extract basic network information automatically

Instead of typing IDs for every element, use the lists stored in the model.

In [ ]:
print('Junctions:', len(wn.junction_name_list))
print('Tanks:', wn.tank_name_list)
print('Pumps:', wn.pump_name_list)
print('Pipes:', len(wn.pipe_name_list))

Select a junction by its position in the list, and inspect its elevation and base demand.

In [ ]:
node_names = wn.junction_name_list

In [ ]:
i = node_names.index('109')
node = wn.get_node(node_names[i])
print('Junction:', node.name)
print('Elevation (m):', node.elevation)
print('Base demand (m3/s):', node.demand_timeseries_list[0].base_value)

💡**Quick exercise:** Change the junction ID above. What changes? What stays the same?

### Use a simple loop to extract information for several junctions

In [ ]:
for i in range(5):
    node = wn.get_node(node_names[i])
    base_demand = node.demand_timeseries_list[0].base_value
    print('Node ID:', node.name, '; Base demand (m3/s):', base_demand)

💡 **Try it:** Change the loop to print the base demands for all junctions. Which junctions have the largest base demands? You can inspect the output for now.

In [ ]:
## Your turn


## 3. Extract and analyze baseline results

Simulation results are DataFrames, with time as rows and node or link IDs as columns.

In [ ]:
print(results_base.node.keys())
print(results_base.link.keys())
results_base.node['pressure'].head()

Select one junction, tank, and pump using the lists. The tank `pressure` result represents water level above the tank bottom.

In [ ]:
selected_node = '109'
tank = wn.tank_name_list[0]
pump = wn.pump_name_list[0]

results_pressure_base = results_base.node['pressure'][selected_node]
results_demand_base = results_base.node['demand'][selected_node] * units
results_tank_base = results_base.node['pressure'][tank]
results_pump_base = results_base.link['flowrate'][pump] * units

time_hours = results_pressure_base.index / 3600

In [ ]:
plt.plot(time_hours, results_pressure_base)
plt.xlabel('Time (hours)')
plt.ylabel('Pressure head (m)')
plt.title('Pressure head at junction ' + selected_node)
plt.show()

💡 **Quick exercise:** Using the same code, plot the tank level and pump flow. What do you notice about their variation over time?

In [ ]:
## Your turn (select different tanks)



In [ ]:
## Your turn (select different pumps)



### A simple performance indicator

In [ ]:
# let's look at minimum pressure

min_pressure = results_base.node['pressure'][wn.junction_name_list].min()
print(min_pressure.head())


In [ ]:
print('Lowest minimum pressure (m):', min_pressure.min())
print('Junction with lowest minimum pressure:', min_pressure.idxmin())

**Discuss:** Why is the minimum pressure across all junctions potentially more useful than pressure at a single selected junction?

## 4. Engineering investigation: uncertain demand

**Scenario:** Actual demand at junction 109 is uncertain. We represent this with a random multiplier between 0.5 and 1.5 of its base demand. First, predict whether the effect on pressure, tank level, and pump flow will be noticeable.

### Change one junction demand and rerun

This follows the same steps as last semester’s stochastic simulation notebook.

In [ ]:
# select node
node = wn.get_node(selected_node)

# get base demand
base_demand = node.demand_timeseries_list[0].base_value

# generate random number between 0.5 to 1.5 from uniform distribution
multiplier = np.random.uniform(0.5, 1.5)
print('Demand multiplier:', multiplier)

# update demand
node.demand_timeseries_list[0].base_value = base_demand * multiplier

# run new simulation
sim = wntr.sim.EpanetSimulator(wn)
results_new = sim.run_sim()

### Compare baseline and uncertain-demand results

In [ ]:
results_pressure_new = results_new.node['pressure'][selected_node]

In [ ]:
plt.plot(time_hours, results_pressure_base, color='gray')
plt.plot(time_hours, results_pressure_new, color='blue')
plt.xlabel('Time (hours)')
plt.ylabel('Pressure head (m)')
plt.title('Pressure at junction ' + selected_node)
plt.legend(['Baseline', 'Modified demand'])
plt.show()

💡 **Quick exercise:** Repeat the same two-line comparison for (a) demand at the selected junction, (b) tank water level, and (c) pump flow.

In [ ]:
## Your turn



## 💡5. Engineering analysis (work in pairs)

1. Did demand increase or decrease? How did pressure at junction 109 respond? Explain the direction of the change.
2. Were the tank level and pump flow affected? Why might the effects be small or large?
3. Find the junction with the lowest minimum pressure **in the modified simulation**. Is it the same junction as in the baseline?
4. Compare the lowest system pressure between the baseline and modified simulations. Is junction 109 a good location for evaluating system vulnerability?
5. Repeat the experiment with a different random multiplier. Does the same conclusion hold?

**Note:** A single simulation with a random multiplier is a *scenario*, not a probability estimate.

In [ ]:
## Your turn



## 💡 6. Final challenge: system-wide demand uncertainty (work in pairs)

So far, we changed demand at only one junction. In practice, demand is uncertain throughout the network.

**Your task:**

1. Start with a fresh Net3 model so you use the **original base demands**.
2. Reuse the `for` loop from Section 2 to go through **all junctions**.
3. At each junction, multiply its original base demand by a **different random number between 0.8 and 1.2** using `np.random.uniform(0.8, 1.2)`.
4. Run the simulation and compare the results with the original baseline.
5. Compare the tank levels and pump flows at **all tanks and pumps**.

**Hint:** You already know how to get a junction with `wn.get_node(...)`, read its base demand, and update it. Combine those commands with the loop from Section 2.

**Note:** Before rerunning the analysis, reload the original network (`wn = wntr.network.WaterNetworkModel('Net3.inp')`). Otherwise, demand changes will accumulate on top of the modifications made in previous runs.

**Discuss:** Does one random realization tell us how *likely* a pressure violation is? What would we need to do to estimate that?
